# EMR Event-Prediction — Architecture Sweep Analysis

Visualisation of the architecture optimisation results on MIMIC-IV.

- Per-experiment ledger: `results/results-architecture optimization.tsv`
- Narrative report: `status.md`
- Final checkpoints: `emr_model/checkpoints/`

Final best model: **M-256** (commit `5496c9e`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.2,
})

## Architecture sweep — `results.tsv`

Five architectures evaluated. The final-pick row for each architecture is selected below (e.g. L-384's row uses the within-size tuned version; M-256 uses the canonical retrain that the deployed checkpoints come from).

In [ ]:
import re

df = pd.read_csv('results/results-architecture optimization.tsv', sep='\t')
for col in ['outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Tag rows by the architecture token that follows the leading "arch " keyword.
# Anchoring on the prefix prevents false matches from comparison phrases later
# in the description (e.g. "vs M-256;" elsewhere in the text).
_ARCH_RE = re.compile(r'^\s*arch\s+([A-Za-z0-9\-]+)')
_RENAMES = {
    'M-256-QA-proper': 'M-256-QA',     # the properly-tokenized run is the canonical M-256-QA
    'M-256-QA':        'M-256-QA (stale tokenizer)',
    'L-384-v2':        'L-384',        # within-size tuned final
    'L-384':           'L-384 (initial)',
}

def tag(desc):
    m = _ARCH_RE.match(str(desc))
    if not m:
        return str(desc)[:20]
    raw = m.group(1)
    if 'retrain' in str(desc):
        return 'M-256'                  # M-256 canonical-retrain row
    return _RENAMES.get(raw, raw)

df['arch'] = df['description'].apply(tag)
df[['commit', 'arch', 'outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb', 'status']]

In [ ]:
# Final-pick rows: one per architecture in the headline comparison.
# - M-256: canonical retrain (5496c9e) — same config as initial run.
# - L-384: within-size tuned final version (v2).
# - M-256-QA: the properly-tokenized run.
# - Stale/intermediate rows and XL-512 (no usable result) are dropped from the headline chart.
final_order = ['S-128', 'M-256', 'M-256-deep', 'L-384', 'M-256-QA']
present     = [a for a in final_order if a in set(df['arch'])]
missing     = [a for a in final_order if a not in set(df['arch'])]
if missing:
    print(f'[note] architectures not present in this TSV — skipped: {missing}')

final = (
    df[df['arch'].isin(present)]
    .drop_duplicates('arch', keep='last')
    .set_index('arch')
    .loc[present]
)
final[['commit', 'outcome_auroc', 'outcome_auprc', 'onset_mae_hrs', 'peak_vram_gb', 'status']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
labels = final.index.tolist()
best   = final['outcome_auroc'].idxmax()
colors = ['#2ecc71' if a == best else '#7f8c8d' for a in labels]

for ax, col, ylabel, fmt in [
    (axes[0], 'outcome_auroc', 'AUROC  \u2191',         '{:.3f}'),
    (axes[1], 'outcome_auprc', 'AUPRC  \u2191',         '{:.3f}'),
    (axes[2], 'onset_mae_hrs', 'Onset MAE (h)  \u2193', '{:.2f}'),
]:
    bars = ax.bar(labels, final[col], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(ylabel, fontsize=11)
    for b, v in zip(bars, final[col]):
        ax.text(b.get_x() + b.get_width() / 2, v, ' ' + fmt.format(v),
                ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', rotation=20)

axes[0].set_title(f'Architecture sweep \u2014 best: {best} (highlighted)', fontsize=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
# Capacity vs accuracy frontier.
param_counts = {
    'S-128': 1.67, 'M-256': 6.41, 'M-256-deep': 9.31, 'L-384': 20.78, 'M-256-QA': 6.42,
}
final['params_M'] = final.index.map(param_counts)

fig, ax = plt.subplots(figsize=(8, 5))
for arch, row in final.iterrows():
    if pd.isna(row['params_M']):
        continue
    c = '#2ecc71' if arch == best else '#7f8c8d'
    ax.scatter(row['params_M'], row['outcome_auroc'],
               s=160, c=c, edgecolor='black', linewidth=0.6, zorder=3)
    ax.annotate(arch, (row['params_M'], row['outcome_auroc']),
                textcoords='offset points', xytext=(8, 6), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('Parameters (millions, log scale)')
ax.set_ylabel('AUROC')
ax.set_title('Capacity vs accuracy — predictive signal saturates at M-256', fontsize=11, loc='left')
plt.tight_layout()
plt.show()

## Final summary

See `status.md` for the full narrative report, per-outcome breakdown, k-day scan, and reproducibility table.

In [ ]:
best_row = final.loc[best]
print(f'Final best:        {best}')
print(f'  commit:          {best_row["commit"]}')
print(f'  AUROC:           {best_row["outcome_auroc"]:.4f}')
print(f'  AUPRC:           {best_row["outcome_auprc"]:.4f}')
print(f'  Onset MAE (h):   {best_row["onset_mae_hrs"]:.2f}')
print(f'  Peak VRAM (GB):  {best_row["peak_vram_gb"]:.2f}')
print(f'  Parameters:      {param_counts[best]:.2f} M')